In [83]:
import warnings
warnings.filterwarnings('ignore')
from pandas import read_csv
from yfinance import download
from pathlib import Path
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
path = Path.cwd().resolve().parent / 'Pipelines' / 'data' / 'pipelines' / 'b3_indices_segmentos_setoriais' / 'processed' / 'transform_1' / 'IBEP.csv'
df_composicao_carteira_indice = read_csv(path)
tickers = df_composicao_carteira_indice['Código'].apply(lambda x: x + '.SA').tolist()

param = {
    'period': '10y',
    'interval': '1d',
}

df = download(tickers, **param, auto_adjust=False)['Adj Close']

[*********************100%***********************]  76 of 76 completed


In [85]:
df = df.pct_change().dropna()
df = df.replace([np.inf, -np.inf], 0) # Transformando inf em zeros

metricas = df.std().to_frame(name='ret_std')
metricas['ret_mean'] = df.mean()
metricas.tail(3)

,ret_std,ret_mean
Ticker,,
VIVT3.SA,0.017711,0.002535
WEGE3.SA,0.016806,0.004077
YDUQ3.SA,0.029598,0.000946


In [86]:
from sklearn.preprocessing import StandardScaler

x = metricas[['ret_std', 'ret_mean']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(x)

In [87]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

metricas['cluster'] = kmeans.fit_predict(X_scaled)
metricas['cluster'].value_counts()

cluster
0    45
1    21
3     9
2     1
Name: count, dtype: int64

In [88]:
df_plot = metricas.reset_index()
df_plot.tail(3)

,Ticker,ret_std,ret_mean,cluster
73,VIVT3.SA,0.017711,0.002535,0
74,WEGE3.SA,0.016806,0.004077,0
75,YDUQ3.SA,0.029598,0.000946,1


In [94]:
ticker_destacado = 'KLBN11.SA'


# =====================
# BASE SCATTER
# =====================

fig = px.scatter(
    df_plot,
    x='ret_mean',
    y='ret_std',
    color='cluster',
    hover_name='Ticker',
    custom_data=['cluster'],
    render_mode='svg',  # melhora resolução vetorial
    template='plotly_white',
    height=650,
    title='Desvio padrão médio x Retorno médio — Agrupamento por Cluster',
    labels={
        'ret_mean': 'Retorno médio',
        'ret_std': 'Desvio padrão (risco)',
        'cluster': 'Cluster'
    }
)

# =====================
# MELHORIA VISUAL DOS PONTOS
# =====================

fig.update_traces(
    marker=dict(
        size=8,
        line=dict(width=0.8, color='black')
    ),
    opacity=0.85,
    hovertemplate=
        '<b>%{hovertext}</b><br>' +
        'Retorno médio: %{x:.4f}<br>' +
        'Desvio padrão: %{y:.4f}<br>' +
        'Cluster: %{customdata[0]}<extra></extra>'
)


# =====================
# LINHAS DE REFERÊNCIA NA MÉDIA GLOBAL
# =====================

fig.add_vline(
    x=df_plot['ret_mean'].mean(),
    line_width=1,
    line_dash="dot"
)

fig.add_hline(
    y=df_plot['ret_std'].mean(),
    line_width=1,
    line_dash="dot"
)


# =====================
# DESTAQUE DO TICKER
# =====================

df_highlight = df_plot.loc[df_plot['Ticker'].eq(ticker_destacado)]

fig.add_trace(
    go.Scatter(
        x=df_highlight['ret_mean'],
        y=df_highlight['ret_std'],
        mode='markers+text',
        marker=dict(
            size=18,
            symbol='star',
            line=dict(width=1.5, color='black')
        ),
        text=df_highlight['Ticker'],
        textposition='top center',
        name=f'Destaque: {ticker_destacado}',
        hovertemplate=
            '<b>%{text}</b><br>' +
            'Retorno médio: %{x:.4f}<br>' +
            'Desvio padrão: %{y:.4f}<extra></extra>'
    )
)


# =====================
# LAYOUT FINAL
# =====================

fig.update_layout(
    title=dict(
        text='Desvio padrão médio x Retorno médio — Agrupamento por Cluster',
        x=0.5
    ),
    legend_title=None,
    template='plotly_white',
    height=650
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)


fig.show()


In [92]:
metricas.loc[metricas['cluster'] == 0]

,ret_std,ret_mean,cluster
Ticker,,,
ABEV3.SA,0.015186,0.005626,0
ALOS3.SA,0.017372,0.003504,0
AXIA6.SA,0.017744,0.004876,0
B3SA3.SA,0.021869,0.005107,0
BBDC3.SA,0.015842,0.002577,0
BBDC4.SA,0.017428,0.002384,0
BPAC11.SA,0.020529,0.003372,0
BRAP4.SA,0.016588,0.005749,0
BRAV3.SA,0.022148,0.003027,0


In [93]:
list(metricas.loc[metricas['cluster'] == 0].index)

['ABEV3.SA',
 'ALOS3.SA',
 'AXIA6.SA',
 'B3SA3.SA',
 'BBDC3.SA',
 'BBDC4.SA',
 'BPAC11.SA',
 'BRAP4.SA',
 'BRAV3.SA',
 'CPFE3.SA',
 'CPLE5.SA',
 'CURY3.SA',
 'EGIE3.SA',
 'EMBJ3.SA',
 'ENEV3.SA',
 'ENGI11.SA',
 'EQTL3.SA',
 'FLRY3.SA',
 'GGBR4.SA',
 'GOAU4.SA',
 'IGTI11.SA',
 'IRBR3.SA',
 'ISAE4.SA',
 'ITSA4.SA',
 'ITUB4.SA',
 'KLBN11.SA',
 'MOTV3.SA',
 'MRVE3.SA',
 'MULT3.SA',
 'PRIO3.SA',
 'PSSA3.SA',
 'RADL3.SA',
 'RENT3.SA',
 'SANB11.SA',
 'SBSP3.SA',
 'SLCE3.SA',
 'SUZB3.SA',
 'TAEE11.SA',
 'TIMS3.SA',
 'UGPA3.SA',
 'USIM5.SA',
 'VALE3.SA',
 'VBBR3.SA',
 'VIVT3.SA',
 'WEGE3.SA']